In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
import glob
from datetime import datetime
import matplotlib.dates as mdates
from matplotlib.font_manager import FontProperties

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

def read_segment_data(segment_folder):
    """
    读取分段数据文件夹中的所有CSV文件
    
    参数:
    segment_folder: 分段数据文件夹路径
    
    返回:
    文件信息列表
    """
    print(f"=== 读取分段数据 ===")
    print(f"数据文件夹: {segment_folder}")
    
    # 获取所有CSV文件
    csv_pattern = os.path.join(segment_folder, "*.csv")
    csv_files = glob.glob(csv_pattern)
    
    if not csv_files:
        print(f"在 {segment_folder} 中未找到CSV文件")
        return []
    
    # 解析文件名信息
    file_info_list = []
    for file_path in csv_files:
        filename = os.path.basename(file_path)
        
        # 解析文件名格式: 20240610_S01_IC2406_RSI_HIGH.csv
        try:
            parts = filename.replace('.csv', '').split('_')
            if len(parts) >= 4:
                date_str = parts[0]
                segment_id = parts[1]
                contract = parts[2]
                condition_type = '_'.join(parts[3:])  # 处理可能有下划线的条件类型
                
                file_info = {
                    'file_path': file_path,
                    'filename': filename,
                    'date': date_str,
                    'segment_id': segment_id,
                    'contract': contract,
                    'condition_type': condition_type
                }
                file_info_list.append(file_info)
        except Exception as e:
            print(f"解析文件名失败: {filename}, 错误: {str(e)}")
            continue
    
    # 按日期和段号排序
    file_info_list.sort(key=lambda x: (x['date'], x['segment_id']))
    
    print(f"找到 {len(file_info_list)} 个分段文件")
    return file_info_list

def plot_segment_data(file_info, output_folder):
    """
    为单个分段数据创建价格走势图
    
    参数:
    file_info: 文件信息字典
    output_folder: 图片输出文件夹
    """
    try:
        # 读取数据
        df = pd.read_csv(file_info['file_path'], encoding='utf-8-sig')
        
        if len(df) == 0:
            print(f"  警告: {file_info['filename']} 文件为空")
            return False
        
        # 确保必要的列存在
        required_cols = ['最新价', '最高价', '最低价']
        missing_cols = [col for col in required_cols if col not in df.columns]
        if missing_cols:
            print(f"  警告: {file_info['filename']} 缺少必要列: {missing_cols}")
            return False
        
        # 处理时间列
        if 'datetime' in df.columns:
            df['datetime'] = pd.to_datetime(df['datetime'])
            time_col = 'datetime'
        elif '最后修改时间' in df.columns:
            # 如果没有datetime列，尝试从交易日和最后修改时间构建
            df['datetime'] = pd.to_datetime(df['交易日'].astype(str) + ' ' + df['最后修改时间'].astype(str))
            time_col = 'datetime'
        else:
            # 如果没有时间列，使用索引作为x轴
            df['index'] = range(len(df))
            time_col = 'index'
        
        # 确保价格列为数值类型
        for col in ['最新价', '最高价', '最低价']:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        
        # 移除价格为NaN的行
        df = df.dropna(subset=['最新价', '最高价', '最低价'])
        
        if len(df) == 0:
            print(f"  警告: {file_info['filename']} 清洗后无有效数据")
            return False
        
        # 计算统计信息
        current_price = df['最新价']
        high_price = df['最高价']
        low_price = df['最低价']
        
        overall_high = high_price.max()
        overall_low = low_price.min()
        start_price = current_price.iloc[0]
        end_price = current_price.iloc[-1]
        price_change = end_price - start_price
        price_change_pct = (price_change / start_price) * 100
        
        # 找到最高点和最低点的位置
        high_idx = high_price.idxmax()
        low_idx = low_price.idxmin()
        
        # 创建图形
        fig, ax = plt.subplots(figsize=(12, 8))
        
        # 绘制价格线
        if time_col == 'datetime':
            x_data = df[time_col]
            # 格式化时间轴
            ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
            ax.xaxis.set_major_locator(mdates.MinuteLocator(interval=30))
            plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)
        else:
            x_data = df[time_col]
            ax.set_xlabel('数据点序号')
        
        # 绘制价格线
        ax.plot(x_data, current_price, label='最新价', linewidth=2, color='blue', alpha=0.8)
        ax.plot(x_data, high_price, label='最高价', linewidth=1, color='red', alpha=0.6)
        ax.plot(x_data, low_price, label='最低价', linewidth=1, color='green', alpha=0.6)
        
        # 填充最高价和最低价之间的区域
        ax.fill_between(x_data, high_price, low_price, alpha=0.2, color='gray', label='价格区间')
        
        # 标记全局最高点和最低点
        ax.scatter(x_data.iloc[high_idx], overall_high, color='red', s=100, marker='^', 
                  label=f'最高点: {overall_high:.2f}', zorder=5)
        ax.scatter(x_data.iloc[low_idx], overall_low, color='green', s=100, marker='v', 
                  label=f'最低点: {overall_low:.2f}', zorder=5)
        
        # 添加标注
        ax.annotate(f'最高: {overall_high:.2f}', 
                   xy=(x_data.iloc[high_idx], overall_high),
                   xytext=(10, 10), textcoords='offset points',
                   bbox=dict(boxstyle='round,pad=0.3', fc='red', alpha=0.7),
                   arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0'))
        
        ax.annotate(f'最低: {overall_low:.2f}', 
                   xy=(x_data.iloc[low_idx], overall_low),
                   xytext=(10, -20), textcoords='offset points',
                   bbox=dict(boxstyle='round,pad=0.3', fc='green', alpha=0.7),
                   arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0'))
        
        # 设置图形标题和标签
        title = (f"{file_info['date']} {file_info['segment_id']} - {file_info['contract']} "
                f"({file_info['condition_type']})\n"
                f"价格变化: {price_change:+.2f} ({price_change_pct:+.2f}%) | "
                f"数据点数: {len(df)} | 价格区间: {overall_low:.2f} - {overall_high:.2f}")
        
        ax.set_title(title, fontsize=12, pad=20)
        ax.set_ylabel('价格')
        if time_col == 'datetime':
            ax.set_xlabel('时间')
        
        # 设置网格
        ax.grid(True, alpha=0.3)
        
        # 设置图例
        ax.legend(loc='best')
        
        # 调整布局
        plt.tight_layout()
        
        # 保存图片
        img_filename = f"{file_info['date']}_{file_info['segment_id']}_{file_info['contract']}_{file_info['condition_type']}.png"
        img_path = os.path.join(output_folder, img_filename)
        plt.savefig(img_path, dpi=300, bbox_inches='tight')
        plt.close()
        
        print(f"  ✓ {file_info['filename']} -> {img_filename}")
        print(f"    数据点: {len(df)}, 价格变化: {price_change:+.2f} ({price_change_pct:+.2f}%)")
        
        return True
        
    except Exception as e:
        print(f"  ✗ 处理 {file_info['filename']} 时出错: {str(e)}")
        return False

def create_summary_plot(file_info_list, output_folder):
    """
    创建所有分段的汇总图
    """
    if len(file_info_list) == 0:
        return
    
    print(f"\n=== 创建汇总统计图 ===")
    
    # 收集所有分段的统计信息
    summary_data = []
    
    for file_info in file_info_list:
        try:
            df = pd.read_csv(file_info['file_path'], encoding='utf-8-sig')
            if len(df) > 0:
                # 确保价格列为数值类型
                for col in ['最新价', '最高价', '最低价']:
                    if col in df.columns:
                        df[col] = pd.to_numeric(df[col], errors='coerce')
                
                df = df.dropna(subset=['最新价'])
                
                if len(df) > 0:
                    start_price = df['最新价'].iloc[0]
                    end_price = df['最新价'].iloc[-1]
                    price_change_pct = ((end_price - start_price) / start_price) * 100
                    
                    summary_data.append({
                        'segment': f"{file_info['date']}_{file_info['segment_id']}",
                        'contract': file_info['contract'],
                        'condition_type': file_info['condition_type'],
                        'data_points': len(df),
                        'price_change_pct': price_change_pct,
                        'high': df['最高价'].max() if '最高价' in df.columns else df['最新价'].max(),
                        'low': df['最低价'].min() if '最低价' in df.columns else df['最新价'].min()
                    })
        except Exception as e:
            continue
    
    if len(summary_data) == 0:
        print("没有有效的汇总数据")
        return
    
    summary_df = pd.DataFrame(summary_data)
    
    # 创建汇总图
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))
    
    # 1. 价格变化百分比分布
    ax1.hist(summary_df['price_change_pct'], bins=20, alpha=0.7, color='blue', edgecolor='black')
    ax1.set_title('价格变化百分比分布')
    ax1.set_xlabel('价格变化 (%)')
    ax1.set_ylabel('频数')
    ax1.grid(True, alpha=0.3)
    ax1.axvline(x=0, color='red', linestyle='--', alpha=0.8)
    
    # 2. 不同条件类型的数量
    condition_counts = summary_df['condition_type'].value_counts()
    ax2.bar(condition_counts.index, condition_counts.values, alpha=0.7)
    ax2.set_title('信号类型分布')
    ax2.set_xlabel('信号类型')
    ax2.set_ylabel('数量')
    ax2.tick_params(axis='x', rotation=45)
    ax2.grid(True, alpha=0.3)
    
    # 3. 数据点数量分布
    ax3.hist(summary_df['data_points'], bins=20, alpha=0.7, color='green', edgecolor='black')
    ax3.set_title('分段数据点数量分布')
    ax3.set_xlabel('数据点数量')
    ax3.set_ylabel('频数')
    ax3.grid(True, alpha=0.3)
    
    # 4. 价格变化 vs 数据点数量散点图
    colors = {'RSI_HIGH': 'red', 'RSI_LOW': 'green', 'MA_ALIGNMENT': 'blue'}
    for condition in summary_df['condition_type'].unique():
        condition_data = summary_df[summary_df['condition_type'] == condition]
        ax4.scatter(condition_data['data_points'], condition_data['price_change_pct'], 
                   alpha=0.7, label=condition, color=colors.get(condition, 'gray'))
    
    ax4.set_title('数据点数量 vs 价格变化')
    ax4.set_xlabel('数据点数量')
    ax4.set_ylabel('价格变化 (%)')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    ax4.axhline(y=0, color='red', linestyle='--', alpha=0.8)
    
    plt.tight_layout()
    
    # 保存汇总图
    summary_img_path = os.path.join(output_folder, "分段数据汇总统计.png")
    plt.savefig(summary_img_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"  ✓ 汇总统计图已保存: 分段数据汇总统计.png")
    
    # 打印统计信息
    print(f"\n=== 汇总统计 ===")
    print(f"总分段数: {len(summary_df)}")
    print(f"信号类型分布:")
    for condition, count in condition_counts.items():
        print(f"  {condition}: {count}")
    print(f"平均价格变化: {summary_df['price_change_pct'].mean():.2f}%")
    print(f"价格上涨分段: {(summary_df['price_change_pct'] > 0).sum()}")
    print(f"价格下跌分段: {(summary_df['price_change_pct'] < 0).sum()}")
    print(f"平均数据点数: {summary_df['data_points'].mean():.0f}")

def main():
    """
    主函数：读取分段数据并生成可视化图表
    """
    # 设置路径
    segment_folder = "/Users/wook/Desktop/因子挖掘/Sample data/raw/期货分段数据"
    output_folder = "/Users/wook/Desktop/因子挖掘/Sample data/charts"
    
    # 确保输出文件夹存在
    os.makedirs(output_folder, exist_ok=True)
    
    print(f"=== 分段数据可视化工具 ===")
    print(f"输入文件夹: {segment_folder}")
    print(f"输出文件夹: {output_folder}")
    
    # 读取所有分段文件信息
    file_info_list = read_segment_data(segment_folder)
    
    if not file_info_list:
        print("没有找到有效的分段文件")
        return
    
    # 为每个分段创建图表
    print(f"\n=== 开始生成图表 ===")
    success_count = 0
    failed_count = 0
    
    for i, file_info in enumerate(file_info_list, 1):
        print(f"[{i}/{len(file_info_list)}] 处理: {file_info['filename']}")
        
        if plot_segment_data(file_info, output_folder):
            success_count += 1
        else:
            failed_count += 1
    
    # 创建汇总统计图
    create_summary_plot(file_info_list, output_folder)
    
    print(f"\n=== 处理完成 ===")
    print(f"成功生成图表: {success_count}")
    print(f"处理失败: {failed_count}")
    print(f"图表保存位置: {output_folder}")
    print(f"")
    print(f"生成的图表包括:")
    print(f"  - 每个分段的价格走势图 ({success_count} 张)")
    print(f"  - 分段数据汇总统计图 (1 张)")

if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.font_manager import FontProperties
import os
import glob
from datetime import datetime
import warnings

# 设置中文字体和图表样式
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']  # 支持中文显示
plt.rcParams['axes.unicode_minus'] = False  # 正常显示负号
plt.style.use('seaborn-v0_8')  # 使用现代化样式
warnings.filterwarnings('ignore')

def find_price_columns(df):
    """自动检测价格列"""
    ask_price_col = None
    bid_price_col = None
    
    # 寻找asks和bids价格列
    for col in df.columns:
        if 'asks[0].price' in col or 'asks_0_price' in col or (col.startswith('asks') and 'price' in col):
            ask_price_col = col
        elif 'bids[0].price' in col or 'bids_0_price' in col or (col.startswith('bids') and 'price' in col):
            bid_price_col = col
    
    # 如果没找到标准列名，尝试其他可能的列名
    if ask_price_col is None:
        potential_ask_cols = [col for col in df.columns if 'ask' in col.lower() and 'price' in col.lower()]
        if potential_ask_cols:
            ask_price_col = potential_ask_cols[0]
    
    if bid_price_col is None:
        potential_bid_cols = [col for col in df.columns if 'bid' in col.lower() and 'price' in col.lower()]
        if potential_bid_cols:
            bid_price_col = potential_bid_cols[0]
    
    return ask_price_col, bid_price_col

def process_snapshot_data(file_path):
    """处理snapshot数据文件"""
    try:
        # 读取CSV文件
        df = pd.read_csv(file_path)
        
        # 检测价格列
        ask_price_col, bid_price_col = find_price_columns(df)
        
        if ask_price_col is None or bid_price_col is None:
            print(f"警告: 无法在文件 {file_path} 中找到价格列")
            print(f"可用列: {df.columns.tolist()}")
            return None
        
        # 转换时间戳
        if 'timestamp' in df.columns:
            df['datetime'] = pd.to_datetime(df['timestamp'], unit='us')
        elif 'local_timestamp' in df.columns:
            df['datetime'] = pd.to_datetime(df['local_timestamp'], unit='us')
        else:
            print(f"警告: 无法在文件 {file_path} 中找到时间戳列")
            return None
        
        # 计算中间价格
        df['mid_price'] = (df[ask_price_col] + df[bid_price_col]) / 2
        df['ask_price'] = df[ask_price_col]
        df['bid_price'] = df[bid_price_col]
        
        # 按时间排序
        df = df.sort_values('datetime')
        
        return df[['datetime', 'mid_price', 'ask_price', 'bid_price']]
        
    except Exception as e:
        print(f"处理文件 {file_path} 时出错: {e}")
        return None

def plot_price_data(df, file_name, output_dir):
    """绘制价格图表"""
    if df is None or len(df) == 0:
        return
    
    # 创建图表
    fig, ax = plt.subplots(figsize=(15, 8))
    
    # 绘制价格线
    ax.plot(df['datetime'], df['mid_price'], label='中间价格', color='blue', linewidth=1.5, alpha=0.8)
    ax.plot(df['datetime'], df['ask_price'], label='卖价', color='red', linewidth=0.8, alpha=0.6)
    ax.plot(df['datetime'], df['bid_price'], label='买价', color='green', linewidth=0.8, alpha=0.6)
    
    # 填充买卖价差区域
    ax.fill_between(df['datetime'], df['ask_price'], df['bid_price'], 
                    alpha=0.2, color='gray', label='买卖价差')
    
    # 标记最高点和最低点
    max_idx = df['mid_price'].idxmax()
    min_idx = df['mid_price'].idxmin()
    
    max_price = df.loc[max_idx, 'mid_price']
    min_price = df.loc[min_idx, 'mid_price']
    max_time = df.loc[max_idx, 'datetime']
    min_time = df.loc[min_idx, 'datetime']
    
    # 标记最高点
    ax.scatter([max_time], [max_price], color='red', s=100, zorder=5, 
               marker='^', label=f'最高点: {max_price:.2f}')
    ax.annotate(f'最高点\n{max_price:.2f}\n{max_time.strftime("%H:%M:%S")}', 
                xy=(max_time, max_price), xytext=(10, 10),
                textcoords='offset points', fontsize=9,
                bbox=dict(boxstyle='round,pad=0.3', facecolor='red', alpha=0.7),
                arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0'))
    
    # 标记最低点
    ax.scatter([min_time], [min_price], color='green', s=100, zorder=5, 
               marker='v', label=f'最低点: {min_price:.2f}')
    ax.annotate(f'最低点\n{min_price:.2f}\n{min_time.strftime("%H:%M:%S")}', 
                xy=(min_time, min_price), xytext=(10, -10),
                textcoords='offset points', fontsize=9,
                bbox=dict(boxstyle='round,pad=0.3', facecolor='green', alpha=0.7),
                arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0'))
    
    # 设置图表格式
    ax.set_title(f'BTC价格走势图 - {file_name}', fontsize=16, fontweight='bold', pad=20)
    ax.set_xlabel('时间', fontsize=12)
    ax.set_ylabel('价格 (USDT)', fontsize=12)
    
    # 格式化x轴时间显示
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
    ax.xaxis.set_major_locator(mdates.MinuteLocator(interval=5))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
    
    # 添加网格
    ax.grid(True, alpha=0.3, linestyle='--')
    
    # 添加图例
    ax.legend(loc='upper left', frameon=True, shadow=True)
    
    # 添加统计信息文本框
    price_range = max_price - min_price
    stats_text = f"""统计信息:
    最高价: {max_price:.2f}
    最低价: {min_price:.2f}
    价格区间: {price_range:.2f}
    数据点数: {len(df):,}
    时间跨度: {(df['datetime'].max() - df['datetime'].min()).total_seconds():.0f}秒"""
    
    ax.text(0.02, 0.98, stats_text, transform=ax.transAxes, fontsize=10,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    # 调整布局
    plt.tight_layout()
    
    # 保存图表
    output_path = os.path.join(output_dir, f"{file_name}_price_chart.png")
    plt.savefig(output_path, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"图表已保存: {output_path}")
    
    # 显示图表（可选）
    # plt.show()
    
    # 关闭图表释放内存
    plt.close()
    
    return {
        'file_name': file_name,
        'max_price': max_price,
        'min_price': min_price,
        'max_time': max_time,
        'min_time': min_time,
        'price_range': price_range,
        'data_points': len(df),
        'time_span_seconds': (df['datetime'].max() - df['datetime'].min()).total_seconds()
    }

def main():
    # 配置路径
    snapshot_data_path = "/Users/wook/Downloads/因子挖掘/Sample data/raw/筛选BTC分段数据/book25_snapshot_1s"
    output_charts_path = "/Users/wook/Downloads/因子挖掘/Sample data/charts"
    
    # 创建输出图表文件夹
    os.makedirs(output_charts_path, exist_ok=True)
    
    # 获取所有snapshot文件
    snapshot_files = glob.glob(os.path.join(snapshot_data_path, "*_snapshot.csv"))
    
    if not snapshot_files:
        print(f"在路径 {snapshot_data_path} 中没有找到snapshot文件")
        return
    
    print(f"找到 {len(snapshot_files)} 个snapshot文件")
    
    # 存储所有文件的统计信息
    all_stats = []
    
    # 处理每个文件
    for i, file_path in enumerate(snapshot_files, 1):
        file_name = os.path.basename(file_path).replace('_snapshot.csv', '')
        print(f"正在处理文件 {i}/{len(snapshot_files)}: {file_name}")
        
        # 处理数据
        df = process_snapshot_data(file_path)
        
        if df is not None:
            # 绘制图表
            stats = plot_price_data(df, file_name, output_charts_path)
            if stats:
                all_stats.append(stats)
        else:
            print(f"跳过文件: {file_name}")
    
    # 保存统计汇总
    if all_stats:
        stats_df = pd.DataFrame(all_stats)
        stats_output_path = os.path.join(output_charts_path, "price_statistics_summary.csv")
        stats_df.to_csv(stats_output_path, index=False, encoding='utf-8-sig')
        print(f"\n统计汇总已保存: {stats_output_path}")
        
        # 显示整体统计
        print(f"\n整体统计:")
        print(f"  处理文件数: {len(all_stats)}")
        print(f"  平均价格区间: {stats_df['price_range'].mean():.2f}")
        print(f"  最大价格区间: {stats_df['price_range'].max():.2f}")
        print(f"  最小价格区间: {stats_df['price_range'].min():.2f}")
        print(f"  平均数据点数: {stats_df['data_points'].mean():.0f}")
        print(f"  平均时间跨度: {stats_df['time_span_seconds'].mean():.0f} 秒")
    
    print(f"\n所有图表已生成完成！图表保存在: {output_charts_path}")

if __name__ == "__main__":
    main()

### h5版

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import os
import glob
from datetime import datetime
import warnings

# 设置中文字体和图表样式
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.style.use('seaborn-v0_8')
warnings.filterwarnings('ignore')

def find_price_columns(df):
    """自动检测价格列"""
    ask_price_col = None
    bid_price_col = None
    
    # 寻找asks和bids价格列
    for col in df.columns:
        if 'asks[0].price' in col or 'asks_0_price' in col or (col.startswith('asks') and 'price' in col):
            ask_price_col = col
        elif 'bids[0].price' in col or 'bids_0_price' in col or (col.startswith('bids') and 'price' in col):
            bid_price_col = col
    
    # 如果没找到标准列名，尝试其他可能的列名
    if ask_price_col is None:
        potential_ask_cols = [col for col in df.columns if 'ask' in col.lower() and 'price' in col.lower()]
        if potential_ask_cols:
            ask_price_col = potential_ask_cols[0]
    
    if bid_price_col is None:
        potential_bid_cols = [col for col in df.columns if 'bid' in col.lower() and 'price' in col.lower()]
        if potential_bid_cols:
            bid_price_col = potential_bid_cols[0]
    
    return ask_price_col, bid_price_col

def process_h5_snapshot_data(file_path):
    """处理H5格式的snapshot数据文件"""
    try:
        # 读取H5文件
        df = pd.read_hdf(file_path, key='snapshot')
        
        # 检测价格列
        ask_price_col, bid_price_col = find_price_columns(df)
        
        if ask_price_col is None or bid_price_col is None:
            print(f"警告: 无法在文件 {file_path} 中找到价格列")
            print(f"可用列: {df.columns.tolist()}")
            return None
        
        # 处理时间戳 - H5文件中可能已经有datetime索引
        if 'datetime' in df.columns:
            df['datetime'] = pd.to_datetime(df['datetime'])
        elif df.index.name == 'datetime' or isinstance(df.index, pd.DatetimeIndex):
            df = df.reset_index()
            if 'datetime' not in df.columns:
                df['datetime'] = df.index
        elif 'timestamp' in df.columns:
            df['datetime'] = pd.to_datetime(df['timestamp'], unit='us')
        else:
            print(f"警告: 无法在文件 {file_path} 中找到时间戳列")
            return None
        
        # 计算中间价格
        df['mid_price'] = (df[ask_price_col] + df[bid_price_col]) / 2
        df['ask_price'] = df[ask_price_col]
        df['bid_price'] = df[bid_price_col]
        
        # 按时间排序
        df = df.sort_values('datetime')
        
        return df[['datetime', 'mid_price', 'ask_price', 'bid_price']]
        
    except Exception as e:
        print(f"处理文件 {file_path} 时出错: {e}")
        return None

def plot_price_data(df, file_name, output_dir):
    """绘制价格图表"""
    if df is None or len(df) == 0:
        return
    
    # 创建图表
    fig, ax = plt.subplots(figsize=(15, 8))
    
    # 绘制价格线
    ax.plot(df['datetime'], df['mid_price'], label='中间价格', color='blue', linewidth=1.5, alpha=0.8)
    ax.plot(df['datetime'], df['ask_price'], label='卖价', color='red', linewidth=0.8, alpha=0.6)
    ax.plot(df['datetime'], df['bid_price'], label='买价', color='green', linewidth=0.8, alpha=0.6)
    
    # 填充买卖价差区域
    ax.fill_between(df['datetime'], df['ask_price'], df['bid_price'], 
                    alpha=0.2, color='gray', label='买卖价差')
    
    # 标记最高点和最低点
    max_idx = df['mid_price'].idxmax()
    min_idx = df['mid_price'].idxmin()
    
    max_price = df.loc[max_idx, 'mid_price']
    min_price = df.loc[min_idx, 'mid_price']
    max_time = df.loc[max_idx, 'datetime']
    min_time = df.loc[min_idx, 'datetime']
    
    # 标记最高点
    ax.scatter([max_time], [max_price], color='red', s=100, zorder=5, 
               marker='^', label=f'最高点: {max_price:.2f}')
    ax.annotate(f'最高点\n{max_price:.2f}\n{max_time.strftime("%H:%M:%S")}', 
                xy=(max_time, max_price), xytext=(10, 10),
                textcoords='offset points', fontsize=9,
                bbox=dict(boxstyle='round,pad=0.3', facecolor='red', alpha=0.7),
                arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0'))
    
    # 标记最低点
    ax.scatter([min_time], [min_price], color='green', s=100, zorder=5, 
               marker='v', label=f'最低点: {min_price:.2f}')
    ax.annotate(f'最低点\n{min_price:.2f}\n{min_time.strftime("%H:%M:%S")}', 
                xy=(min_time, min_price), xytext=(10, -10),
                textcoords='offset points', fontsize=9,
                bbox=dict(boxstyle='round,pad=0.3', facecolor='green', alpha=0.7),
                arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0'))
    
    # 判断趋势类型并设置标题颜色
    title_color = 'blue'
    trend_info = ""
    if 'LONG' in file_name or '多头' in file_name:
        title_color = 'green'
        trend_info = " (多头行情)"
    elif 'SHORT' in file_name or '空头' in file_name:
        title_color = 'red'
        trend_info = " (空头行情)"
    
    # 设置图表格式
    ax.set_title(f'BTC价格走势图{trend_info} - {file_name}', 
                fontsize=16, fontweight='bold', pad=20, color=title_color)
    ax.set_xlabel('时间', fontsize=12)
    ax.set_ylabel('价格 (USDT)', fontsize=12)
    
    # 格式化x轴时间显示
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
    ax.xaxis.set_major_locator(mdates.MinuteLocator(interval=5))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
    
    # 添加网格
    ax.grid(True, alpha=0.3, linestyle='--')
    
    # 添加图例
    ax.legend(loc='upper left', frameon=True, shadow=True)
    
    # 添加统计信息文本框
    price_range = max_price - min_price
    price_change_pct = ((max_price - min_price) / min_price) * 100
    stats_text = f"""统计信息:
最高价: {max_price:.2f}
最低价: {min_price:.2f}
价格区间: {price_range:.2f}
价格变幅: {price_change_pct:.2f}%
数据点数: {len(df):,}
时间跨度: {(df['datetime'].max() - df['datetime'].min()).total_seconds():.0f}秒"""
    
    ax.text(0.02, 0.98, stats_text, transform=ax.transAxes, fontsize=10,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    # 调整布局
    plt.tight_layout()
    
    # 保存图表
    output_path = os.path.join(output_dir, f"{file_name}_price_chart.png")
    plt.savefig(output_path, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"图表已保存: {output_path}")
    
    # 关闭图表释放内存
    plt.close()
    
    return {
        'file_name': file_name,
        'max_price': max_price,
        'min_price': min_price,
        'max_time': max_time,
        'min_time': min_time,
        'price_range': price_range,
        'price_change_pct': price_change_pct,
        'data_points': len(df),
        'time_span_seconds': (df['datetime'].max() - df['datetime'].min()).total_seconds()
    }

def main():
    # 配置路径
    snapshot_data_path = "/Users/wook/Downloads/因子挖掘/Sample data/raw/筛选BTC分段数据/book25_snapshot_1s"
    output_charts_path = "/Users/wook/Downloads/因子挖掘/Sample data/charts"
    
    # 创建输出图表文件夹
    os.makedirs(output_charts_path, exist_ok=True)
    
    # 获取所有H5 snapshot文件
    snapshot_files = glob.glob(os.path.join(snapshot_data_path, "*_snapshot.h5"))
    
    if not snapshot_files:
        print(f"在路径 {snapshot_data_path} 中没有找到H5 snapshot文件")
        return
    
    print(f"找到 {len(snapshot_files)} 个H5 snapshot文件")
    
    # 存储所有文件的统计信息
    all_stats = []
    
    # 处理每个文件
    for i, file_path in enumerate(snapshot_files, 1):
        file_name = os.path.basename(file_path).replace('_snapshot.h5', '')
        print(f"正在处理文件 {i}/{len(snapshot_files)}: {file_name}")
        
        # 处理H5数据
        df = process_h5_snapshot_data(file_path)
        
        if df is not None:
            # 绘制图表
            stats = plot_price_data(df, file_name, output_charts_path)
            if stats:
                all_stats.append(stats)
        else:
            print(f"跳过文件: {file_name}")
    
    # 保存统计汇总
    if all_stats:
        stats_df = pd.DataFrame(all_stats)
        
        # 保存为H5格式
        stats_output_path = os.path.join(output_charts_path, "price_statistics_summary.h5")
        stats_df.to_hdf(stats_output_path, key='summary', mode='w', format='table', complevel=9)
        
        # 同时保存CSV格式便于查看
        csv_output_path = os.path.join(output_charts_path, "price_statistics_summary.csv")
        stats_df.to_csv(csv_output_path, index=False, encoding='utf-8-sig')
        
        print(f"\n统计汇总已保存: {stats_output_path}")
        print(f"CSV格式汇总已保存: {csv_output_path}")
        
        # 显示整体统计
        print(f"\n整体统计:")
        print(f"  处理文件数: {len(all_stats)}")
        print(f"  平均价格区间: {stats_df['price_range'].mean():.2f}")
        print(f"  最大价格区间: {stats_df['price_range'].max():.2f}")
        print(f"  最小价格区间: {stats_df['price_range'].min():.2f}")
        print(f"  平均价格变幅: {stats_df['price_change_pct'].mean():.2f}%")
        print(f"  平均数据点数: {stats_df['data_points'].mean():.0f}")
        print(f"  平均时间跨度: {stats_df['time_span_seconds'].mean():.0f} 秒")
        
        # 分别统计多头和空头数据
        long_stats = stats_df[stats_df['file_name'].str.contains('LONG')]
        short_stats = stats_df[stats_df['file_name'].str.contains('SHORT')]
        
        if len(long_stats) > 0:
            print(f"\n多头行情统计:")
            print(f"  文件数: {len(long_stats)}")
            print(f"  平均价格变幅: {long_stats['price_change_pct'].mean():.2f}%")
        
        if len(short_stats) > 0:
            print(f"\n空头行情统计:")
            print(f"  文件数: {len(short_stats)}")
            print(f"  平均价格变幅: {short_stats['price_change_pct'].mean():.2f}%")
    
    print(f"\n所有图表已生成完成！图表保存在: {output_charts_path}")

if __name__ == "__main__":
    main()

找到 190 个H5 snapshot文件
正在处理文件 1/190: SHORT_0073_MA空头
图表已保存: /Users/wook/Downloads/因子挖掘/Sample data/charts/SHORT_0073_MA空头_price_chart.png
正在处理文件 2/190: SHORT_0095_MA空头
图表已保存: /Users/wook/Downloads/因子挖掘/Sample data/charts/SHORT_0095_MA空头_price_chart.png
正在处理文件 3/190: LONG_0091_MA多头
图表已保存: /Users/wook/Downloads/因子挖掘/Sample data/charts/LONG_0091_MA多头_price_chart.png
正在处理文件 4/190: LONG_0077_MA多头
图表已保存: /Users/wook/Downloads/因子挖掘/Sample data/charts/LONG_0077_MA多头_price_chart.png
正在处理文件 5/190: LONG_0049_MA多头
图表已保存: /Users/wook/Downloads/因子挖掘/Sample data/charts/LONG_0049_MA多头_price_chart.png
正在处理文件 6/190: LONG_0012_MA多头
图表已保存: /Users/wook/Downloads/因子挖掘/Sample data/charts/LONG_0012_MA多头_price_chart.png
正在处理文件 7/190: SHORT_0016_MA空头
图表已保存: /Users/wook/Downloads/因子挖掘/Sample data/charts/SHORT_0016_MA空头_price_chart.png
正在处理文件 8/190: SHORT_0028_MA空头
图表已保存: /Users/wook/Downloads/因子挖掘/Sample data/charts/SHORT_0028_MA空头_price_chart.png
正在处理文件 9/190: SHORT_0009_MA空头
图表已保存: /Users/wook/Downloads/因子挖掘/Sa